# 🔍 Sensor Anomaly Detection - Exploratory Data Analysis

This notebook explores the NASA Bearing vibration dataset and demonstrates
the anomaly detection pipeline.

## Contents
1. Data Generation & Loading
2. Signal Visualization
3. Feature Extraction
4. Feature Distribution Analysis
5. Model Training & Evaluation
6. Anomaly Visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.ingestion import NASABearingDataLoader
from src.data.preprocessing import TimeSeriesPreprocessor
from src.features.extraction import FeatureExtractor
from src.models.isolation_forest import IsolationForestDetector
from src.models.lstm_autoencoder import LSTMAutoencoderDetector

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Data Generation

We generate synthetic bearing vibration data that simulates the degradation
pattern observed in real bearings: normal → early degradation → failure.

In [ ]:
# Generate synthetic data
loader = NASABearingDataLoader(data_dir='../data/raw')
data = loader.generate_synthetic_data(n_snapshots=500, n_samples=2048, n_channels=4)

print(f'Data shape: {data.shape}')
print(f'Timestamps: {data.index.get_level_values("timestamp").nunique()}')
print(f'Channels: {list(data.columns)}')
data.head()

## 2. Signal Visualization

Visualize raw vibration signals at different stages of bearing life.

In [ ]:
timestamps = data.index.get_level_values('timestamp').unique()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

stages = [
    ('Normal Operation', timestamps[50]),
    ('Early Degradation', timestamps[400]),
    ('Near Failure', timestamps[480]),
]

for ax, (title, ts) in zip(axes, stages):
    signal = data.loc[ts]['ch1'].values[:512]
    ax.plot(signal, linewidth=0.5, alpha=0.8)
    ax.set_title(f'{title} (RMS: {np.sqrt(np.mean(signal**2)):.4f})', fontsize=12)
    ax.set_ylabel('Amplitude')
    ax.set_ylim(-5, 5)

axes[-1].set_xlabel('Sample')
plt.suptitle('Bearing Vibration Signal at Different Life Stages', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Extraction

Extract time-domain and frequency-domain features from each snapshot.

In [ ]:
extractor = FeatureExtractor(sample_rate=20480)
features_df = extractor.extract_from_snapshots(data)

print(f'Feature matrix shape: {features_df.shape}')
print(f'\nFeature names:')
for i, col in enumerate(features_df.columns, 1):
    print(f'  {i:2d}. {col}')

features_df.describe()

## 4. Feature Distribution Analysis

In [ ]:
# Plot key features over time
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

key_features = ['ch1_rms', 'ch1_kurtosis', 'ch1_spectral_centroid', 'ch1_crest_factor']

for ax, feature in zip(axes, key_features):
    values = features_df[feature].values
    ax.plot(values, linewidth=0.8)
    ax.set_ylabel(feature.replace('ch1_', '').replace('_', ' ').title())
    ax.axvline(x=len(values)*0.7, color='red', linestyle='--', alpha=0.5, label='Degradation start')

axes[0].legend()
axes[-1].set_xlabel('Snapshot Index')
plt.suptitle('Feature Evolution Over Bearing Lifetime', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Model Training & Evaluation

In [ ]:
# Preprocess
preprocessor = TimeSeriesPreprocessor(train_ratio=0.7, normalize=True)
processed = preprocessor.prepare_features(features_df)

print(f'Training samples: {processed.X_train.shape[0]}')
print(f'Test samples: {processed.X_test.shape[0]}')

In [ ]:
# Train Isolation Forest
if_detector = IsolationForestDetector(n_estimators=200, contamination=0.05, random_state=42)
if_detector.fit(processed.X_train)

# Detect anomalies
train_results = if_detector.detect(processed.X_train)
test_results = if_detector.detect(processed.X_test)

print(f'\nIsolation Forest Results:')
print(f'  Train anomalies: {train_results["n_anomalies"]} ({train_results["anomaly_ratio"]:.2%})')
print(f'  Test anomalies: {test_results["n_anomalies"]} ({test_results["anomaly_ratio"]:.2%})')

## 6. Anomaly Visualization

In [ ]:
# Combine all scores
all_scores = np.concatenate([
    if_detector.score_samples(processed.X_train),
    if_detector.score_samples(processed.X_test)
])
all_labels = np.concatenate([train_results['labels'], test_results['labels']])

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Anomaly scores over time
ax = axes[0]
colors = ['green' if l == 1 else 'red' for l in all_labels]
ax.scatter(range(len(all_scores)), all_scores, c=colors, s=10, alpha=0.6)
ax.axvline(x=len(processed.X_train), color='blue', linestyle='--', label='Train/Test split')
ax.set_xlabel('Snapshot Index')
ax.set_ylabel('Anomaly Score')
ax.set_title('Isolation Forest Anomaly Scores')
ax.legend()

# Score distribution
ax = axes[1]
ax.hist(all_scores[all_labels == 1], bins=50, alpha=0.7, label='Normal', color='green')
ax.hist(all_scores[all_labels == -1], bins=50, alpha=0.7, label='Anomaly', color='red')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Count')
ax.set_title('Score Distribution')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# LSTM Autoencoder
seq_length = 20
X_train_seq = preprocessor.create_sequences(processed.X_train, sequence_length=seq_length)
X_test_seq = preprocessor.create_sequences(processed.X_test, sequence_length=seq_length)

print(f'Training sequences: {X_train_seq.shape}')
print(f'Test sequences: {X_test_seq.shape}')

lstm_detector = LSTMAutoencoderDetector(
    n_features=processed.X_train.shape[1],
    hidden_size=32,
    num_layers=2,
    epochs=30,
    batch_size=32,
    patience=10,
    threshold_percentile=95,
    device='cpu'
)

lstm_detector.fit(X_train_seq)
lstm_results = lstm_detector.detect(X_test_seq)

print(f'\nLSTM Autoencoder Results:')
print(f'  Test anomalies: {lstm_results["n_anomalies"]} ({lstm_results["anomaly_ratio"]:.2%})')
print(f'  Threshold: {lstm_results["threshold"]:.6f}')

In [ ]:
# Plot LSTM reconstruction errors
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Reconstruction errors
ax = axes[0]
errors = lstm_results['errors']
colors = ['red' if l == -1 else 'green' for l in lstm_results['labels']]
ax.scatter(range(len(errors)), errors, c=colors, s=10, alpha=0.6)
ax.axhline(y=lstm_results['threshold'], color='orange', linestyle='--', label=f'Threshold ({lstm_results["threshold"]:.4f})')
ax.set_xlabel('Sequence Index')
ax.set_ylabel('Reconstruction Error')
ax.set_title('LSTM Autoencoder Reconstruction Errors')
ax.legend()

# Training loss curve
ax = axes[1]
ax.plot(lstm_detector.train_losses, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('Training Loss Curve')

plt.tight_layout()
plt.show()

## Summary

Both models successfully detect the degradation and failure patterns:

- **Isolation Forest**: Detects anomalies based on statistical isolation in feature space
- **LSTM Autoencoder**: Detects anomalies based on reconstruction error of temporal patterns

The LSTM model captures temporal dependencies and may detect early degradation
patterns that the Isolation Forest misses, while Isolation Forest is faster to train
and more interpretable.